In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# -- Step 1: Load checkpointed data ------------------------------------------
checkpoint_dir = "checkpoints"

df_hospitalA = pd.read_parquet(f"{checkpoint_dir}/hospitalA_imputed_clean.parquet")
df_hospitalB = pd.read_parquet(f"{checkpoint_dir}/hospitalB_imputed_clean.parquet")

print(f"Hospital A: {df_hospitalA.shape[0]:,} rows, {df_hospitalA['Patient_ID'].nunique()} patients")
print(f"Hospital B: {df_hospitalB.shape[0]:,} rows, {df_hospitalB['Patient_ID'].nunique()} patients")

Hospital A: 368,278 rows, 11453 patients
Hospital B: 630,502 rows, 18680 patients


In [3]:
# -- Step 2: Combine ----------------------------------------------------------
combined = pd.concat([df_hospitalA, df_hospitalB], ignore_index=True)
print(f"\nCombined: {combined.shape[0]:,} rows, {combined['Patient_ID'].nunique()} patients")
print(combined['Hospital'].value_counts())


Combined: 998,780 rows, 30133 patients
Hospital
B    630502
A    368278
Name: count, dtype: int64


In [4]:
# -- Step 3: Build a patient-level lookup table for stratified splitting ----
# One row per patient: whether they ever had sepsis, and which hospital
patient_info = combined.groupby('Patient_ID').agg(
    SepsisEver=('SepsisLabel', 'max'),
    Hospital=('Hospital', 'first')
).reset_index()

# Combine Hospital + SepsisEver into one stratify key (so both are balanced)
patient_info['strata'] = patient_info['Hospital'] + '_' + patient_info['SepsisEver'].astype(str)

print(f"\nPatient-level strata distribution:")
print(patient_info['strata'].value_counts())


Patient-level strata distribution:
strata
B_0    17850
A_0    10492
A_1      961
B_1      830
Name: count, dtype: int64


In [5]:
# -- Step 4: Split PATIENTS (not rows!) into train/val/test -----------------
# 70% train, 15% val, 15% test
train_ids, temp_ids = train_test_split(
    patient_info['Patient_ID'],
    test_size=0.30,
    stratify=patient_info['strata'],
    random_state=42
)

# Need strata for the remaining patients to split val/test correctly
temp_info = patient_info[patient_info['Patient_ID'].isin(temp_ids)]

val_ids, test_ids = train_test_split(
    temp_info['Patient_ID'],
    test_size=0.50,  # 50% of the remaining 30% = 15% val, 15% test
    stratify=temp_info['strata'],
    random_state=42
)

# -- Step 5: Create the actual row-level dataframes --------------------------
df_train = combined[combined['Patient_ID'].isin(train_ids)].copy()
df_val   = combined[combined['Patient_ID'].isin(val_ids)].copy()
df_test  = combined[combined['Patient_ID'].isin(test_ids)].copy()

In [6]:
# -- Step 6: Verify split quality ---------------------------------------------
def summarize_split(df, name):
    n_patients = df['Patient_ID'].nunique()
    n_rows     = df.shape[0]
    sepsis_patients = df[df['SepsisLabel'] == 1]['Patient_ID'].nunique()
    hosp_counts = df.groupby('Hospital')['Patient_ID'].nunique()
    print(f"\n{name}:")
    print(f"   Patients : {n_patients:,}")
    print(f"   Rows     : {n_rows:,}")
    print(f"   Sepsis patients : {sepsis_patients} ({sepsis_patients/n_patients*100:.2f}%)")
    print(f"   Hospital breakdown : {dict(hosp_counts)}")

print(f"\n{'='*60}")
print("  Train / Val / Test Split Summary")
print(f"{'='*60}")
summarize_split(df_train, "Train")
summarize_split(df_val,   "Validation")
summarize_split(df_test,  "Test")

# -- Step 7: Sanity check - no patient overlap across splits ----------------
assert len(set(train_ids) & set(val_ids)) == 0, "Train/Val overlap!"
assert len(set(train_ids) & set(test_ids)) == 0, "Train/Test overlap!"
assert len(set(val_ids) & set(test_ids)) == 0, "Val/Test overlap!"
print("\nNo patient overlap between splits - leakage-free.")


  Train / Val / Test Split Summary

Train:
   Patients : 21,093
   Rows     : 701,019
   Sepsis patients : 1254 (5.95%)
   Hospital breakdown : {'A': np.int64(8017), 'B': np.int64(13076)}

Validation:
   Patients : 4,520
   Rows     : 149,040
   Sepsis patients : 269 (5.95%)
   Hospital breakdown : {'A': np.int64(1718), 'B': np.int64(2802)}

Test:
   Patients : 4,520
   Rows     : 148,721
   Sepsis patients : 268 (5.93%)
   Hospital breakdown : {'A': np.int64(1718), 'B': np.int64(2802)}

No patient overlap between splits - leakage-free.


In [7]:
import os

# -- Create a directory for the split datasets -------------------------------
split_dir = "checkpoints/splits"
os.makedirs(split_dir, exist_ok=True)

# -- Save as Parquet (recommended) -------------------------------------------
df_train.to_parquet(f"{split_dir}/train.parquet", index=False)
df_val.to_parquet(f"{split_dir}/val.parquet", index=False)
df_test.to_parquet(f"{split_dir}/test.parquet", index=False)

# -- Also save as CSV (for quick inspection) ---------------------------------
df_train.to_csv(f"{split_dir}/train.csv", index=False)
df_val.to_csv(f"{split_dir}/val.csv", index=False)
df_test.to_csv(f"{split_dir}/test.csv", index=False)

print("Saved train/val/test splits:")
print(f"   {split_dir}/train.parquet  ({df_train.shape[0]:,} rows, {df_train['Patient_ID'].nunique()} patients)")
print(f"   {split_dir}/val.parquet    ({df_val.shape[0]:,} rows, {df_val['Patient_ID'].nunique()} patients)")
print(f"   {split_dir}/test.parquet   ({df_test.shape[0]:,} rows, {df_test['Patient_ID'].nunique()} patients)")

# -- Also save the patient ID lists separately (useful for reproducibility) -
pd.Series(sorted(train_ids)).to_csv(f"{split_dir}/train_patient_ids.csv", index=False, header=['Patient_ID'])
pd.Series(sorted(val_ids)).to_csv(f"{split_dir}/val_patient_ids.csv", index=False, header=['Patient_ID'])
pd.Series(sorted(test_ids)).to_csv(f"{split_dir}/test_patient_ids.csv", index=False, header=['Patient_ID'])

print(f"\nPatient ID lists saved for reproducibility:")
print(f"   {split_dir}/train_patient_ids.csv")
print(f"   {split_dir}/val_patient_ids.csv")
print(f"   {split_dir}/test_patient_ids.csv")

Saved train/val/test splits:
   checkpoints/splits/train.parquet  (701,019 rows, 21093 patients)
   checkpoints/splits/val.parquet    (149,040 rows, 4520 patients)
   checkpoints/splits/test.parquet   (148,721 rows, 4520 patients)

Patient ID lists saved for reproducibility:
   checkpoints/splits/train_patient_ids.csv
   checkpoints/splits/val_patient_ids.csv
   checkpoints/splits/test_patient_ids.csv
